# AI-ENG-201 HW2 Analysis Notebook
**Author:** Fatima Alibabayeva  
**Course:** AI-ENG-201 Machine Learning, AI Academy  
**Semester:** Fall 2026

In [3]:
import numpy as np
import os
import sys
import matplotlib
matplotlib.use('Agg')  # headless render
import matplotlib.pyplot as plt

# figures/ qovluğunu yaradiriq (varsa saxlayir)
os.makedirs('figures', exist_ok=True)

# src modullarini import etmek ucun path elave edirik
sys.path.insert(0, '.')

np.random.seed(42)
print("Setup complete. NumPy:", np.__version__)

Setup complete. NumPy: 2.4.1


## Part 1: Linear Regression
### 1.1 — OLS Closed-Form: Sanity Check vs sklearn

In [5]:
from src.linear_regression import LinearRegression
from sklearn.linear_model import LinearRegression as SklearnLR

np.random.seed(42)
X_sanity = np.random.randn(100, 5)
y_sanity = np.random.randn(100)

my_lr = LinearRegression()
my_lr.fit(X_sanity, y_sanity)

sk_lr = SklearnLR()
sk_lr.fit(X_sanity, y_sanity)

diff = np.max(np.abs(my_lr.predict(X_sanity) - sk_lr.predict(X_sanity)))
print(f"Max absolute difference vs sklearn: {diff:.2e}")
assert diff < 1e-9, "Sanity check FAILED!"
print("Sanity check PASSED (diff < 1e-9) ✓")

ModuleNotFoundError: No module named 'src'

### 1.2 — Gradient Descent: MSE vs Iteration

In [ ]:
from src.linear_regression_gd import LinearRegressionGD

np.random.seed(42)
X_gd = np.random.randn(1000, 10)
y_gd = np.random.randn(1000)

lrs = [0.001, 0.01, 0.1]
plt.figure(figsize=(8, 5))

conv_table = {}
for lr in lrs:
    np.random.seed(42)
    model = LinearRegressionGD(lr=lr, max_iter=1000, tol=1e-6)
    model.fit(X_gd, y_gd)
    iters = len(model.mse_history)
    final_mse = model.mse_history[-1]
    conv_table[lr] = (iters, final_mse)
    plt.plot(model.mse_history, label=f'η={lr} (conv. @ iter {iters})')

plt.xlabel('Iteration')
plt.ylabel('MSE')
plt.title('Gradient Descent: MSE vs Iteration (N=1000, p=10, seed=42)')
plt.legend()
plt.yscale('log')
plt.grid(True)
plt.tight_layout()
plt.savefig('figures/gd_mse.pdf', bbox_inches='tight')
plt.show()

print("\nConvergence summary:")
print(f"{'η':<8} {'Iters':>8} {'Final MSE':>12}")
print("-" * 30)
for lr, (iters, mse) in conv_table.items():
    converged = "converged" if iters < 1000 else "NOT converged"
    print(f"{lr:<8} {iters:>8} {mse:>12.6f}  ({converged})")

In [ ]:
# GD vs closed-form comparison (seed 42, N=1000, p=10)
np.random.seed(42)
X_cf = np.random.randn(1000, 10)
y_cf = np.random.randn(1000)

cf_model = LinearRegression()
cf_model.fit(X_cf, y_cf)

gd_model = LinearRegressionGD(lr=0.1, max_iter=1000, tol=1e-6)
gd_model.fit(X_cf, y_cf)

max_diff = np.max(np.abs(cf_model.w[1:] - gd_model.w[1:]))
print(f"GD vs Closed-form max weight difference: {max_diff:.2e}")
assert max_diff < 1e-4, "GD did not match closed-form within 1e-4!"
print("Match confirmed (< 1e-4) ✓")

### 1.3 — Polynomial Fitting on California Housing

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split

data_ca = fetch_california_housing()
X_full = data_ca.data
y_full = data_ca.target
print(f"California Housing loaded: {X_full.shape}")
print(f"Features: {list(data_ca.feature_names)}")

In [ ]:
# Yalniz MedInc feature-i (sutun 0) istifade edirik
x_medinc = X_full[:, 0]

# Task spec: N_train = 100, N_val = 100, random_state=42
X_train_1d, X_val_1d, y_train, y_val = train_test_split(
    x_medinc, y_full, train_size=100, test_size=100, random_state=42
)
X_train = X_train_1d.reshape(-1, 1)
X_val = X_val_1d.reshape(-1, 1)
print(f"Train: {X_train.shape}, Val: {X_val.shape}")

In [ ]:
train_mses = []
val_mses = []

for d in range(1, 13):
    # polynomial design matrix: [x, x^2, ..., x^d]
    X_poly_train = np.column_stack([X_train ** i for i in range(1, d + 1)])
    X_poly_val   = np.column_stack([X_val   ** i for i in range(1, d + 1)])

    # standardize edirik — yuksek derecelerde numerik stabillik ucun vacibdir
    mu = X_poly_train.mean(axis=0)
    sd = X_poly_train.std(axis=0)
    sd[sd == 0] = 1  # sifira bolunmemek ucun

    X_poly_train_std = (X_poly_train - mu) / sd
    X_poly_val_std   = (X_poly_val - mu) / sd

    model = LinearRegression()
    model.fit(X_poly_train_std, y_train)

    train_mses.append(np.mean((model.predict(X_poly_train_std) - y_train) ** 2))
    val_mses.append(np.mean((model.predict(X_poly_val_std) - y_val) ** 2))

best_d = np.argmin(val_mses) + 1
print(f"Best degree (lowest Val MSE): d = {best_d}")
print(f"Val MSE at d={best_d}: {val_mses[best_d-1]:.4f}")
print(f"\nAll Val MSEs:")
for d, v in enumerate(val_mses, 1):
    marker = " <-- best" if d == best_d else ""
    print(f"  d={d:2d}: val_mse={v:.4f}{marker}")

In [ ]:
degrees = list(range(1, 13))

plt.figure(figsize=(8, 5))
plt.plot(degrees, train_mses, 'o-', label='Train MSE')
plt.plot(degrees, val_mses, 's-', label='Val MSE')
plt.xlabel('Polynomial Degree (d)')
plt.ylabel('MSE')
plt.title('Polynomial Fitting: Train vs Validation MSE (MedInc feature, N=100)')
plt.legend()
plt.grid(True)
plt.yscale('log')  # log scale — yuksek derecelerde cox boyuk deyerler var
plt.tight_layout()
plt.savefig('figures/poly_fit.pdf', bbox_inches='tight')
plt.show()
print("Saved: figures/poly_fit.pdf")

### 1.5 — Prediction Intervals (d=5 and d=12)

In [ ]:
from scipy import stats

for d, fname in [(5, 'figures/pred_intervals_d5.pdf'), (12, 'figures/pred_intervals_d12.pdf')]:
    X_poly_train = np.column_stack([X_train ** i for i in range(1, d + 1)])
    X_poly_val   = np.column_stack([X_val   ** i for i in range(1, d + 1)])

    # standardize (fit zamanı öğrənilən mean/std ile)
    mu = X_poly_train.mean(axis=0)
    sd = X_poly_train.std(axis=0); sd[sd == 0] = 1
    X_poly_train_s = (X_poly_train - mu) / sd
    X_poly_val_s   = (X_poly_val - mu) / sd

    N_tr, p_tr = X_poly_train_s.shape

    model = LinearRegression()
    model.fit(X_poly_train_s, y_train)
    y_pred_tr  = model.predict(X_poly_train_s)
    y_pred_val = model.predict(X_poly_val_s)

    # noise variance estimate: sigma^2 = RSS / (N - p)
    sigma2 = np.sum((y_train - y_pred_tr) ** 2) / (N_tr - p_tr)

    # (X^T X)^{-1} ucun bias column elave edirik
    Xt_ = np.hstack([np.ones((N_tr, 1)), X_poly_train_s])
    Xv_ = np.hstack([np.ones((len(X_poly_val_s), 1)), X_poly_val_s])
    XtX_inv = np.linalg.inv(Xt_.T @ Xt_)

    t_crit = stats.t.ppf(0.975, df=N_tr - p_tr)

    # hər validation nöqtəsi ucun 95% PI genişliyi
    intervals = np.array([
        t_crit * np.sqrt(sigma2) * np.sqrt(1 + x @ XtX_inv @ x)
        for x in Xv_
    ])

    print(f"d={d}: sigma2={sigma2:.4f}, mean PI width={intervals.mean():.4f}")

    plt.figure(figsize=(8, 5))
    plt.scatter(y_pred_val, y_val, alpha=0.6, label='True vs Predicted', zorder=3)
    plt.errorbar(y_pred_val, y_val, yerr=intervals, fmt='none', alpha=0.25,
                 color='orange', label='95% Prediction Interval')
    plt.xlabel('Predicted value')
    plt.ylabel('True value')
    plt.title(f'Prediction Intervals (d={d}, California Housing MedInc)')
    plt.legend()
    plt.tight_layout()
    plt.savefig(fname, bbox_inches='tight')
    plt.show()
    print(f"Saved: {fname}")

## Part 2: Regularization & Tuning
### 2.1 + 2.2 — Ridge & Lasso Coefficient Paths (Diabetes dataset)

In [ ]:
from sklearn.datasets import load_diabetes
from src.ridge_regression import RidgeRegression
from src.lasso_regression import LassoRegression

np.random.seed(42)

data_d = load_diabetes()
X_d = data_d.data
y_d = data_d.target
feature_names_d = list(data_d.feature_names)

# standardize edirik ki penalty hamiya bərabər tətbiq olunsun
X_d_mean = X_d.mean(axis=0)
X_d_std  = X_d.std(axis=0); X_d_std[X_d_std == 0] = 1
X_d_sc   = (X_d - X_d_mean) / X_d_std

# Ridge sanity check vs sklearn
from sklearn.linear_model import Ridge
sk_r = Ridge(alpha=1.0, fit_intercept=True)
sk_r.fit(X_d_sc, y_d)
my_r = RidgeRegression(lambda_=1.0)
my_r.fit(X_d_sc, y_d)
ridge_diff = np.max(np.abs(my_r.predict(X_d_sc) - sk_r.predict(X_d_sc)))
print(f"Ridge vs sklearn diff: {ridge_diff:.2e}  (should be < 1e-9)")
assert ridge_diff < 1e-9
print("Ridge sanity check PASSED ✓")

In [ ]:
# lambda deyerlerini logarithmik araligda aliriq: 1e-3 to 1e2
lambdas = np.logspace(-3, 2, 50)

ridge_coefs = []
lasso_coefs = []

for lam in lambdas:
    ridge = RidgeRegression(lambda_=lam)
    ridge.fit(X_d_sc, y_d)
    ridge_coefs.append(ridge.w[1:])  # w[0] intercept-dir, skip edirik

    lasso = LassoRegression(lambda_=lam)
    lasso.fit(X_d_sc, y_d)
    lasso_coefs.append(lasso.w.copy())

ridge_coefs = np.array(ridge_coefs)
lasso_coefs = np.array(lasso_coefs)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Ridge coefficient path
for j in range(X_d_sc.shape[1]):
    axes[0].plot(np.log10(lambdas), ridge_coefs[:, j], label=feature_names_d[j])
axes[0].set_xlabel('log₁₀(λ)')
axes[0].set_ylabel('Coefficient value')
axes[0].set_title('Ridge Coefficient Path (Diabetes, standardized)')
axes[0].legend(fontsize=7, loc='upper right')
axes[0].axhline(0, color='k', linewidth=0.5, linestyle='--')
axes[0].grid(True, alpha=0.3)

# Lasso coefficient path
for j in range(X_d_sc.shape[1]):
    axes[1].plot(np.log10(lambdas), lasso_coefs[:, j], label=feature_names_d[j])
axes[1].set_xlabel('log₁₀(λ)')
axes[1].set_ylabel('Coefficient value')
axes[1].set_title('Lasso Coefficient Path (Diabetes, standardized)')
axes[1].legend(fontsize=7, loc='upper right')
axes[1].axhline(0, color='k', linewidth=0.5, linestyle='--')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/coef_paths_diabetes.pdf', bbox_inches='tight')
plt.show()
print("Saved: figures/coef_paths_diabetes.pdf")

# Sparsity analizi
print("\nLasso sparsity at different λ values:")
for lam in [0.001, 0.1, 1.0, 10.0, 50.0]:
    l = LassoRegression(lambda_=lam)
    l.fit(X_d_sc, y_d)
    nz = np.sum(np.abs(l.w) > 1e-4)
    print(f"  λ={lam:5.3f}: {nz}/10 features nonzero")

### 2.3 — Hyperparameter Search (Grid vs Random) — California Housing

In [ ]:
from sklearn.model_selection import KFold
import time

np.random.seed(42)

# California Housing full dataset (artiq yüklənib yuxarıda)
X_ca = data_ca.data
y_ca = data_ca.target

# featurelari standardize edirik
X_ca_mean = X_ca.mean(axis=0)
X_ca_std  = X_ca.std(axis=0); X_ca_std[X_ca_std == 0] = 1
X_ca_sc   = (X_ca - X_ca_mean) / X_ca_std

kf = KFold(n_splits=5, shuffle=True, random_state=42)

def cv_rmse(lambdas_list, X, y, kf):
    """5-fold CV RMSE for each lambda value."""
    results = []
    for lam in lambdas_list:
        fold_rmses = []
        for train_idx, val_idx in kf.split(X):
            X_tr, X_v = X[train_idx], X[val_idx]
            y_tr, y_v = y[train_idx], y[val_idx]
            model = RidgeRegression(lambda_=lam)
            model.fit(X_tr, y_tr)
            y_pred = model.predict(X_v)
            rmse = np.sqrt(np.mean((y_v - y_pred) ** 2))
            fold_rmses.append(rmse)
        results.append(np.mean(fold_rmses))
    return np.array(results)

# Grid search: log araliginda bərabər aralikli 20 lambda
grid_lambdas = np.logspace(-4, 4, 20)
t0 = time.time()
grid_rmses = cv_rmse(grid_lambdas, X_ca_sc, y_ca, kf)
grid_time = time.time() - t0
best_grid_lam = grid_lambdas[np.argmin(grid_rmses)]
best_grid_rmse = np.min(grid_rmses)

# Random search: eyni araligda 20 random lambda (log-uniform)
rng = np.random.RandomState(42)
rand_lambdas = np.power(10, rng.uniform(-4, 4, 20))
t0 = time.time()
rand_rmses = cv_rmse(rand_lambdas, X_ca_sc, y_ca, kf)
rand_time = time.time() - t0
best_rand_lam = rand_lambdas[np.argmin(rand_rmses)]
best_rand_rmse = np.min(rand_rmses)

print(f"Grid Search  — time: {grid_time:.3f}s | best λ: {best_grid_lam:.5f} | best RMSE: {best_grid_rmse:.4f}")
print(f"Random Search — time: {rand_time:.3f}s | best λ: {best_rand_lam:.5f} | best RMSE: {best_rand_rmse:.4f}")

In [ ]:
plt.figure(figsize=(8, 5))
plt.semilogx(grid_lambdas, grid_rmses, 'o-', label=f'Grid Search (best λ={best_grid_lam:.4f})')
plt.semilogx(rand_lambdas, rand_rmses, 's--', label=f'Random Search (best λ={best_rand_lam:.4f})', alpha=0.8)
plt.xlabel('λ (log scale)')
plt.ylabel('5-fold CV RMSE')
plt.title('Hyperparameter Search: Grid vs Random (Ridge, California Housing)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/hp_search.pdf', bbox_inches='tight')
plt.show()
print("Saved: figures/hp_search.pdf")

### 2.4 — Learning Curves (OLS vs Ridge)

In [ ]:
np.random.seed(42)

# validation seti datanin 20%-i — sabit saxlayiriq
n_total = X_ca_sc.shape[0]
n_val = int(0.2 * n_total)
X_val_lc  = X_ca_sc[-n_val:]
y_val_lc  = y_ca[-n_val:]
X_pool    = X_ca_sc[:-n_val]
y_pool    = y_ca[:-n_val]

train_sizes = [100, 500, 1000, 5000, len(X_pool)]

ols_train_rmses, ols_val_rmses     = [], []
ridge_train_rmses, ridge_val_rmses = [], []

for n in train_sizes:
    X_tr = X_pool[:n]
    y_tr = y_pool[:n]

    # OLS
    ols = LinearRegression()
    ols.fit(X_tr, y_tr)
    ols_train_rmses.append(np.sqrt(np.mean((y_tr - ols.predict(X_tr)) ** 2)))
    ols_val_rmses.append(np.sqrt(np.mean((y_val_lc - ols.predict(X_val_lc)) ** 2)))

    # Ridge — best lambda grid search-dan
    ridge = RidgeRegression(lambda_=best_grid_lam)
    ridge.fit(X_tr, y_tr)
    ridge_train_rmses.append(np.sqrt(np.mean((y_tr - ridge.predict(X_tr)) ** 2)))
    ridge_val_rmses.append(np.sqrt(np.mean((y_val_lc - ridge.predict(X_val_lc)) ** 2)))

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, ols_train_rmses, 'o-',  color='steelblue',  label='OLS Train RMSE')
plt.plot(train_sizes, ols_val_rmses,   'o--', color='steelblue',  label='OLS Val RMSE')
plt.plot(train_sizes, ridge_train_rmses, 's-',  color='darkorange', label='Ridge Train RMSE')
plt.plot(train_sizes, ridge_val_rmses,   's--', color='darkorange', label='Ridge Val RMSE')
plt.xlabel('Training set size ($N_{train}$)')
plt.ylabel('RMSE')
plt.title('Learning Curves: OLS vs Ridge (California Housing, standardized)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('figures/learning_curves.pdf', bbox_inches='tight')
plt.show()
print("Saved: figures/learning_curves.pdf")

## Part 3: Logistic Regression & Naive Bayes
### 3.1 — Logistic Regression Sanity Check

In [ ]:
from src.logistic_regression import LogisticRegression
from sklearn.linear_model import LogisticRegression as SklearnLR

np.random.seed(42)

# linearly separable binary data
X_bin = np.random.randn(300, 5)
y_bin = (X_bin[:, 0] + X_bin[:, 1] > 0).astype(int)

# bizim implementasiya
my_lr = LogisticRegression(lr=0.5, lambda_=0.0, max_iter=2000)
my_lr.fit(X_bin, y_bin)
my_preds = my_lr.predict(X_bin)
my_proba = my_lr.predict_proba(X_bin)

# sklearn reference (C=inf ≈ no regularization)
sk_lr2 = SklearnLR(C=1e9, solver='lbfgs', max_iter=2000, random_state=42)
sk_lr2.fit(X_bin, y_bin)
sk_preds = sk_lr2.predict(X_bin)

acc_my = np.mean(my_preds == y_bin)
acc_sk = np.mean(sk_preds == y_bin)
agree  = np.mean(my_preds == sk_preds)
print(f"My LR accuracy:      {acc_my:.4f}")
print(f"Sklearn LR accuracy: {acc_sk:.4f}")
print(f"Prediction agreement: {agree:.4f}")

# proba range check
assert np.all(my_proba >= 0) and np.all(my_proba <= 1)
print("Proba in [0,1]: ✓")

### 3.2 — Multiclass One-vs-Rest on Wine Dataset

In [ ]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
import matplotlib

np.random.seed(42)

wine = load_wine()
X_w = wine.data
y_w = wine.target

# standardize
X_w_mean = X_w.mean(axis=0)
X_w_std  = X_w.std(axis=0); X_w_std[X_w_std == 0] = 1
X_w_sc   = (X_w - X_w_mean) / X_w_std

X_tr_w, X_te_w, y_tr_w, y_te_w = train_test_split(X_w_sc, y_w, test_size=0.2, random_state=42)

classes = np.unique(y_w)
K = len(classes)

# One-vs-Rest: hər class ucun ayrıca binary classifier train edirik
classifiers = []
scores = np.zeros((len(X_te_w), K))

for i, c in enumerate(classes):
    # c classini 1, qalanlarini 0 kimi label edirik
    y_tr_bin = (y_tr_w == c).astype(int)
    clf = LogisticRegression(lr=0.1, lambda_=0.01, max_iter=2000)
    clf.fit(X_tr_w, y_tr_bin)
    classifiers.append(clf)
    # test set ucun probability score
    scores[:, i] = clf.predict_proba(X_te_w)

# en yuksek score-a sahib class-i secirik
y_pred_ovr = classes[np.argmax(scores, axis=1)]

acc_ovr = accuracy_score(y_te_w, y_pred_ovr)
cm_ovr = confusion_matrix(y_te_w, y_pred_ovr)
print(f"OvR Logistic Regression Accuracy (Wine): {acc_ovr:.4f}")
print("\nConfusion Matrix:")
print(cm_ovr)

# confusion matrix gorsel
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm_ovr, cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')
ax.set_title('Confusion Matrix — OvR Logistic Regression (Wine)')
ax.set_xticks(range(K)); ax.set_xticklabels(wine.target_names, rotation=15)
ax.set_yticks(range(K)); ax.set_yticklabels(wine.target_names)
for i in range(K):
    for j in range(K):
        ax.text(j, i, cm_ovr[i, j], ha='center', va='center',
                color='white' if cm_ovr[i, j] > cm_ovr.max()/2 else 'black')
plt.tight_layout()
plt.savefig('figures/confusion_matrix_lr.pdf', bbox_inches='tight')
plt.show()
print("Saved: figures/confusion_matrix_lr.pdf")

### 3.3 — Gaussian Naive Bayes vs Logistic Regression (Wine)

In [ ]:
from src.naive_bayes import GaussianNaiveBayes
from sklearn.metrics import precision_score, recall_score, f1_score

np.random.seed(42)

# Naive Bayes
gnb = GaussianNaiveBayes()
gnb.fit(X_tr_w, y_tr_w)
y_pred_nb = gnb.predict(X_te_w)

acc_nb  = accuracy_score(y_te_w, y_pred_nb)
prec_nb = precision_score(y_te_w, y_pred_nb, average='macro', zero_division=0)
rec_nb  = recall_score(y_te_w, y_pred_nb, average='macro', zero_division=0)
f1_nb   = f1_score(y_te_w, y_pred_nb, average='macro', zero_division=0)

# Logistic Regression (OvR — yuxaridaki nəticə)
acc_lr  = acc_ovr
prec_lr = precision_score(y_te_w, y_pred_ovr, average='macro', zero_division=0)
rec_lr  = recall_score(y_te_w, y_pred_ovr, average='macro', zero_division=0)
f1_lr   = f1_score(y_te_w, y_pred_ovr, average='macro', zero_division=0)

print("=== Wine Dataset — Model Comparison ===")
print(f"{'Model':<22} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1':>10}")
print("-" * 65)
print(f"{'Gaussian Naive Bayes':<22} {acc_nb:>10.4f} {prec_nb:>10.4f} {rec_nb:>10.4f} {f1_nb:>10.4f}")
print(f"{'Logistic Reg (OvR)':<22} {acc_lr:>10.4f} {prec_lr:>10.4f} {rec_lr:>10.4f} {f1_lr:>10.4f}")

# proba sum-to-1 check
proba_nb = gnb.predict_proba(X_te_w)
assert np.allclose(proba_nb.sum(axis=1), 1.0, atol=1e-6)
print("\nNB proba sums to 1: ✓")

### 3.4 — Text Classification on 20 Newsgroups

In [ ]:
from src.text_features import BagOfWords, TfidfTransformer
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import f1_score, roc_auc_score

np.random.seed(42)

# yalniz iki class: comp.graphics ve talk.religion.misc
categories = ['comp.graphics', 'talk.religion.misc']
train_news = fetch_20newsgroups(subset='train', categories=categories,
                                remove=('headers', 'footers', 'quotes'))
test_news  = fetch_20newsgroups(subset='test',  categories=categories,
                                remove=('headers', 'footers', 'quotes'))

# BoW: top 5000 sozu vocabulary kimi oyrenirik
bow = BagOfWords(max_features=5000)
X_tr_bow = bow.fit_transform(train_news.data)
X_te_bow = bow.transform(test_news.data)

# TF-IDF: count matrix-i TF-IDF weightlerine ceviririk
tfidf = TfidfTransformer()
X_tr_tfidf = tfidf.fit_transform(X_tr_bow)
X_te_tfidf = tfidf.transform(X_te_bow)

y_tr_text = train_news.target
y_te_text = test_news.target

print(f"Train: {X_tr_tfidf.shape}, Test: {X_te_tfidf.shape}")
print(f"Classes: {train_news.target_names}")

# Logistic Regression
lr_text = LogisticRegression(lr=0.1, lambda_=0.01, max_iter=1000)
lr_text.fit(X_tr_tfidf, y_tr_text)
y_pred_lr_text  = lr_text.predict(X_te_tfidf)
y_proba_lr_text = lr_text.predict_proba(X_te_tfidf)

# Gaussian Naive Bayes
gnb_text = GaussianNaiveBayes()
gnb_text.fit(X_tr_tfidf, y_tr_text)
y_pred_nb_text  = gnb_text.predict(X_te_tfidf)
y_proba_nb_text = gnb_text.predict_proba(X_te_tfidf)[:, 1]  # class 1 probability

f1_lr_text  = f1_score(y_te_text, y_pred_lr_text, average='binary')
auc_lr_text = roc_auc_score(y_te_text, y_proba_lr_text)
f1_nb_text  = f1_score(y_te_text, y_pred_nb_text, average='binary')
auc_nb_text = roc_auc_score(y_te_text, y_proba_nb_text)

print("\n=== 20 Newsgroups Text Classification ===")
print(f"{'Model':<22} {'F1':>8} {'AUC-ROC':>10}")
print("-" * 42)
print(f"{'Logistic Regression':<22} {f1_lr_text:>8.4f} {auc_lr_text:>10.4f}")
print(f"{'Gaussian NB':<22} {f1_nb_text:>8.4f} {auc_nb_text:>10.4f}")

## Part 4: Evaluation & Interpretation
### 4.2 — ROC + Calibration (Text Classification)

In [ ]:
from sklearn.metrics import roc_curve

np.random.seed(42)

fpr_lr, tpr_lr, _ = roc_curve(y_te_text, y_proba_lr_text)
fpr_nb, tpr_nb, _ = roc_curve(y_te_text, y_proba_nb_text)

def calibration_curve_manual(y_true, y_prob, n_bins=10):
    """10-bin reliability diagram: mean predicted prob vs empirical accuracy."""
    bins = np.linspace(0, 1, n_bins + 1)
    mean_probs, mean_accs = [], []
    for i in range(n_bins):
        mask = (y_prob >= bins[i]) & (y_prob < bins[i + 1])
        if mask.sum() > 0:
            mean_probs.append(y_prob[mask].mean())
            mean_accs.append(y_true[mask].mean())
    return np.array(mean_probs), np.array(mean_accs)

mp_lr, ma_lr = calibration_curve_manual(y_te_text, y_proba_lr_text)
mp_nb, ma_nb = calibration_curve_manual(y_te_text, y_proba_nb_text)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# ROC curves
axes[0].plot(fpr_lr, tpr_lr, label=f'Logistic Reg (AUC={auc_lr_text:.3f})')
axes[0].plot(fpr_nb, tpr_nb, label=f'Gaussian NB (AUC={auc_nb_text:.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', label='Random classifier')
axes[0].set_xlabel('False Positive Rate')
axes[0].set_ylabel('True Positive Rate')
axes[0].set_title('ROC Curves — 20 Newsgroups Text Classification')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Calibration (reliability diagram)
axes[1].plot(mp_lr, ma_lr, 'o-', label='Logistic Reg')
axes[1].plot(mp_nb, ma_nb, 's-', label='Gaussian NB')
axes[1].plot([0, 1], [0, 1], 'k--', label='Perfect calibration')
axes[1].set_xlabel('Mean predicted probability')
axes[1].set_ylabel('Empirical accuracy')
axes[1].set_title('Calibration Curve (Reliability Diagram)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/roc_calibration.pdf', bbox_inches='tight')
plt.show()
print("Saved: figures/roc_calibration.pdf")

### 4.1 — Regularization Paths on California Housing

In [ ]:
np.random.seed(42)

# 50 lambda, log 1e-3 to 1e3
lambdas_ca = np.logspace(-3, 3, 50)
ca_feature_names = list(data_ca.feature_names)

ridge_coefs_ca = []
lasso_coefs_ca = []

for lam in lambdas_ca:
    r = RidgeRegression(lambda_=lam)
    r.fit(X_ca_sc, y_ca)
    ridge_coefs_ca.append(r.w[1:])  # intercept skip

    l = LassoRegression(lambda_=lam)
    l.fit(X_ca_sc, y_ca)
    lasso_coefs_ca.append(l.w.copy())

ridge_coefs_ca = np.array(ridge_coefs_ca)
lasso_coefs_ca = np.array(lasso_coefs_ca)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for j in range(X_ca_sc.shape[1]):
    axes[0].plot(np.log10(lambdas_ca), ridge_coefs_ca[:, j], label=ca_feature_names[j])
axes[0].set_xlabel('log₁₀(λ)')
axes[0].set_ylabel('Coefficient value')
axes[0].set_title('Ridge Coefficient Path (California Housing)')
axes[0].legend(fontsize=7)
axes[0].axhline(0, color='k', linewidth=0.5, linestyle='--')
axes[0].grid(True, alpha=0.3)

for j in range(X_ca_sc.shape[1]):
    axes[1].plot(np.log10(lambdas_ca), lasso_coefs_ca[:, j], label=ca_feature_names[j])
axes[1].set_xlabel('log₁₀(λ)')
axes[1].set_ylabel('Coefficient value')
axes[1].set_title('Lasso Coefficient Path (California Housing)')
axes[1].legend(fontsize=7)
axes[1].axhline(0, color='k', linewidth=0.5, linestyle='--')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('figures/reg_paths_ca_housing.pdf', bbox_inches='tight')
plt.show()
print("Saved: figures/reg_paths_ca_housing.pdf")

# AveRooms (2) ve AveBedrms (3) korrelyasiyasi
corr = np.corrcoef(X_ca_sc[:, 2], X_ca_sc[:, 3])[0, 1]
print(f"\nCorrelation AveRooms & AveBedrms: {corr:.4f}")